In [ ]:
import signal

class MonitorLoop:
    def __init__(self):
        self.running = True
        self.last_states = {}
        
        # Registra handler de interrupção
        signal.signal(signal.SIGINT, self.stop)
    
    def stop(self, signum, frame):
        print("\n🛑 Parando monitoramento...")
        self.running = False
    
    def run(self, interval=30):
        """
        Roda monitoramento contínuo
        interval: segundos entre verificações
        """
        print(f"🚀 Monitoramento iniciado - verificação a cada {interval}s")
        print(f"📱 Webhook: {NTFY_WEBHOOK}")
        print(f"📝 Para interromper, pressione Ctrl+C\n")
        
        check_count = 0
        
        while self.running:
            check_count += 1
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            print(f"\n[{check_count}] {timestamp} — Verificando condições...")
            
            for condition_name, condition_config in MONITOR_CONDITIONS.items():
                status, message = check_condition(condition_name, condition_config)
                
                # Verifica se estado mudou
                previous_status = self.last_states.get(condition_name)
                self.last_states[condition_name] = status
                
                # Registra evento
                log_event("check", condition_name, status, message)
                
                # Se status mudou para erro, envia alerta
                if status == "error" and previous_status != "error":
                    print(f"   ⚠️  ALERTA: {message}")
                    success, alert_msg = send_alert("error", f"🚨 Alerta: {condition_name}", message)
                    print(f"      {alert_msg}")
                    log_event("alert", condition_name, "sent", alert_msg)
                else:
                    print(f"   ✓ {message}")
            
            if self.running:
                time.sleep(interval)
        
        print(f"\n✅ Monitoramento encerrado após {check_count} verificações")

# Inicia monitoramento
monitor = MonitorLoop()
# Descomente a linha abaixo para rodar continuamente:
# monitor.run(interval=30)  # Verifica a cada 30 segundos

print("⏸️  Monitoramento está PAUSADO")
print("   Descomente a última linha para iniciar o loop contínuo")

## Loop de Monitoramento Contínuo

Execute esta célula para manter o monitoramento ligado.

In [ ]:
def log_event(event_type, condition, status, message):
    """
    Registra um evento no histórico JSON
    """
    
    # Carrega histórico anterior
    if EVENT_LOG_FILE.exists():
        with open(EVENT_LOG_FILE, 'r') as f:
            events = json.load(f)
    else:
        events = []
    
    # Adiciona novo evento
    event = {
        "timestamp": datetime.now().isoformat(),
        "type": event_type,
        "condition": condition,
        "status": status,
        "message": message
    }
    
    events.append(event)
    
    # Mantém apenas últimos 1000 eventos
    events = events[-1000:]
    
    # Salva histórico
    with open(EVENT_LOG_FILE, 'w') as f:
        json.dump(events, f, indent=2, ensure_ascii=False)
    
    return event

# Teste de logging
event = log_event("check", "nightly_task", "ok", "Tarefa noturna rodou sem erros")
print(f"✅ Evento registrado: {event['timestamp']}")

# Mostra últimos eventos
if EVENT_LOG_FILE.exists():
    with open(EVENT_LOG_FILE, 'r') as f:
        events = json.load(f)
    print(f"\n📋 Total de eventos: {len(events)}")
    if events:
        print(f"Último evento: {events[-1]['timestamp']}")

## 5. Registrar Eventos e Status

Mantém um histórico de todas as verificações e notificações.

In [ ]:
def send_alert(alert_type, title, message):
    """
    Envia alerta via ntfy.sh
    alert_type: "error", "warning", "info"
    """
    
    headers = {
        "Title": title,
        "Priority": "high" if alert_type == "error" else "normal",
        "Tags": "warning" if alert_type == "error" else "info",
    }
    
    full_message = f"[{alert_type.upper()}] {datetime.now().strftime('%H:%M:%S')}\n{message}"
    
    try:
        response = requests.post(
            NTFY_WEBHOOK,
            data=full_message,
            headers=headers,
            timeout=5
        )
        
        if response.status_code == 200:
            return True, "✅ Notificação enviada"
        else:
            return False, f"❌ Erro ao enviar: {response.status_code}"
    
    except Exception as e:
        return False, f"❌ Exceção: {str(e)}"

# Teste de envio
success, msg = send_alert(
    "error",
    "⚠️ Teste de Alerta",
    "Este é um teste do sistema de monitoramento"
)
print(msg)

## 4. Enviar Notificação para o App

Usa ntfy.sh (serviço público) ou seu próprio webhook para enviar alertas.

In [ ]:
def check_condition(condition_name, condition_config):
    """
    Verifica uma condição específica
    Returns: (status, message)
    """
    log_file = condition_config["file"]
    
    if not log_file.exists():
        return "warning", f"Arquivo não encontrado: {log_file.name}"
    
    # Lê o arquivo de log
    with open(log_file, 'r') as f:
        content = f.read()
    
    # Verifica tamanho (arquivo vazio = OK, arquivo grande = problema)
    size = len(content)
    threshold = condition_config["threshold"]
    
    if size > threshold:
        return "error", f"{condition_name}: Erro detectado ({size} bytes)"
    
    return "ok", f"{condition_name}: OK"

# Testa função
for name, config in MONITOR_CONDITIONS.items():
    status, msg = check_condition(name, config)
    print(f"[{status.upper()}] {msg}")

## 3. Verificar Condição e Detectar Eventos

Função que verifica periodicamente se algo aconteceu.

In [ ]:
# Configuração de monitoramento
PROJECT_DIR = Path("/Users/andreluis/Documents/juridico-crm")
LOGS_DIR = PROJECT_DIR / "logs"

# Condições críticas a monitorar
MONITOR_CONDITIONS = {
    "nightly_task": {"file": LOGS_DIR / "launchd_err.log", "threshold": 0},
    "report_delivery": {"file": LOGS_DIR / "report_delivery_err.log", "threshold": 0},
    "database_connection": {"file": LOGS_DIR / "database.log", "threshold": 5},
}

# Webhook do Claude - CONFIGURE COM SEU ENDPOINT
# Você pode usar:
# 1. Twilio WhatsApp API
# 2. Firebase Cloud Messaging
# 3. Um servidor webhook próprio
# 4. Ntfy.sh (serviço público gratuito)

NTFY_TOPIC = "juridico_crm_alerts"  # Tópico único do ntfy.sh
NTFY_WEBHOOK = f"https://ntfy.sh/{NTFY_TOPIC}"

# Histórico de eventos
EVENT_LOG_FILE = LOGS_DIR / "monitor_events.json"

print(f"✅ Configuração carregada")
print(f"   Monitores: {list(MONITOR_CONDITIONS.keys())}")
print(f"   Webhook: {NTFY_WEBHOOK}")

## 2. Configurar Condição de Monitoramento

Definir os parâmetros e webhooks para enviar notificações.

## 1. Importar Bibliotecas

Importar bibliotecas como requests, time e json para monitorar o sistema e chamar APIs.

In [ ]:
import requests
import time
import json
import os
from datetime import datetime
from pathlib import Path

print("✅ Bibliotecas importadas com sucesso")

# Monitor de Alertas — CRM Jurídico

Sistema de monitoramento em tempo real que notifica você no app do Claude no celular quando algo acontece.

**Funcionalidades:**
- ✅ Monitora tarefas noturnas
- ✅ Detecta erros e falhas
- ✅ Envia alertas em tempo real
- ✅ Registra histórico de eventos